# NB 07: Results Synthesis

Publication-quality tables and figures. Best exchange pairs ranked by
risk-adjusted return. Comparison with Werapun et al. (2025).

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.gridspec import GridSpec

from src.config import EXCHANGES, PROCESSED_DIR, FIGURES_DIR

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 200
plt.rcParams['font.size'] = 10

# Load results
bt_results = pd.read_parquet(PROCESSED_DIR / 'backtest_results.parquet')
oos = bt_results[bt_results['sample_type'] == 'out_of_sample'].copy()
print(f'OOS results: {len(oos)} configurations')

## 1. Exchange Pair Rankings

In [ ]:
# Rank by OOS Sharpe ratio, aggregating across parameter configs
if not oos.empty:
    pair_ranking = oos.groupby(['exchange_a', 'exchange_b', 'asset']).agg(
        median_sharpe=('sharpe_ratio', 'median'),
        best_sharpe=('sharpe_ratio', 'max'),
        median_return=('annualized_return', 'median'),
        median_drawdown=('max_drawdown', 'median'),
        total_trades=('n_trades', 'sum'),
        n_configs=('sharpe_ratio', 'count'),
    ).round(4)
    
    pair_ranking['pair_type'] = pair_ranking.index.map(
        lambda x: 'CEX-CEX' if EXCHANGES[x[0]]['type'] == 'CEX' and EXCHANGES[x[1]]['type'] == 'CEX'
        else 'DEX-DEX' if EXCHANGES[x[0]]['type'] == 'DEX' and EXCHANGES[x[1]]['type'] == 'DEX'
        else 'CEX-DEX'
    )
    
    pair_ranking = pair_ranking.sort_values('median_sharpe', ascending=False)
    
    print('Exchange Pair Rankings (by median OOS Sharpe):')
    pair_ranking

## 2. Sharpe Ratio Heatmap

In [ ]:
if not oos.empty:
    for asset in sorted(oos['asset'].unique()):
        asset_data = oos[oos['asset'] == asset]
        if asset_data.empty:
            continue
        
        # Best Sharpe per pair
        best_per_pair = asset_data.groupby(['exchange_a', 'exchange_b'])['sharpe_ratio'].max()
        
        # Create matrix
        exchanges = sorted(set(asset_data['exchange_a'].unique()) | set(asset_data['exchange_b'].unique()))
        matrix = pd.DataFrame(np.nan, index=exchanges, columns=exchanges)
        
        for (ea, eb), sr in best_per_pair.items():
            matrix.loc[ea, eb] = sr
            matrix.loc[eb, ea] = sr
        
        fig, ax = plt.subplots(figsize=(8, 6))
        sns.heatmap(matrix.astype(float), annot=True, fmt='.2f',
                    cmap='RdYlGn', center=0, ax=ax,
                    mask=matrix.isna())
        ax.set_title(f'{asset} Best OOS Sharpe Ratio by Exchange Pair')
        plt.tight_layout()
        plt.savefig(FIGURES_DIR / f'sharpe_heatmap_{asset}.pdf', bbox_inches='tight')
        plt.show()

## 3. Comparison with Werapun et al. (2025)

In [ ]:
# Reference paper results
werapun = {
    'max_return_6mo': 1.159,  # 115.9%
    'max_drawdown': 0.0192,   # 1.92%
    'corr_with_hodl': 0.0,    # ~zero
    'exchanges': ['Binance', 'BitMEX', 'ApolloX', 'Drift'],
    'assets': ['BTC', 'ETH', 'XRP', 'BNB', 'SOL'],
    'strategy': 'CEX/DEX funding rate arbitrage',
    'period': '6 months',
}

print('=== Comparison with Werapun et al. (2025) ===')
print(f'\nReference paper:')
print(f'  Max 6-month return: {werapun["max_return_6mo"]*100:.1f}%')
print(f'  Max drawdown: {werapun["max_drawdown"]*100:.2f}%')
print(f'  Correlation with HODL: ~{werapun["corr_with_hodl"]}')
print(f'  Exchanges: {", ".join(werapun["exchanges"])}')

if not oos.empty:
    our_best = oos.nlargest(1, 'sharpe_ratio').iloc[0]
    print(f'\nOur results (best OOS config):')
    print(f'  Annualized return: {our_best.get("annualized_return", 0)*100:.1f}%')
    print(f'  Max drawdown: {our_best.get("max_drawdown", 0)*100:.2f}%')
    print(f'  Sharpe ratio: {our_best.get("sharpe_ratio", 0):.2f}')
    print(f'  Pair: {our_best.get("exchange_a")}/{our_best.get("exchange_b")} {our_best.get("asset")}')
    
    # Key differences
    print(f'\nKey differences from reference:')
    print(f'  - We use cross-exchange (perp-perp), they used cash-and-carry + cross-exchange')
    print(f'  - Different exchange set (we include OKX, dYdX, Hyperliquid)')
    print(f'  - We model transaction costs and basis risk explicitly')
    print(f'  - 70/30 chronological train/test split')

## 4. Publication Summary Table

In [ ]:
if not oos.empty:
    # Best config per pair (by Sharpe)
    best_per_pair = oos.loc[oos.groupby(['exchange_a', 'exchange_b', 'asset'])['sharpe_ratio'].idxmax()]
    
    summary_cols = {
        'exchange_a': 'Exchange A',
        'exchange_b': 'Exchange B',
        'asset': 'Asset',
        'n_trades': 'N Trades',
        'total_return': 'Total Return',
        'annualized_return': 'Ann. Return',
        'sharpe_ratio': 'Sharpe',
        'sortino_ratio': 'Sortino',
        'max_drawdown': 'Max DD',
        'calmar_ratio': 'Calmar',
        'hit_rate': 'Hit Rate',
    }
    
    available_cols = {k: v for k, v in summary_cols.items() if k in best_per_pair.columns}
    
    pub_table = best_per_pair[list(available_cols.keys())].rename(columns=available_cols)
    pub_table = pub_table.sort_values('Sharpe', ascending=False)
    
    # Format percentages
    for col in ['Total Return', 'Ann. Return', 'Max DD', 'Hit Rate']:
        if col in pub_table.columns:
            pub_table[col] = (pub_table[col] * 100).round(2).astype(str) + '%'
    
    for col in ['Sharpe', 'Sortino', 'Calmar']:
        if col in pub_table.columns:
            pub_table[col] = pub_table[col].round(2)
    
    print('Publication Summary Table (OOS, Best Config per Pair):')
    pub_table

## 5. Key Findings

In [ ]:
if not oos.empty:
    print('=== KEY FINDINGS ===')
    print()
    
    # 1. Profitability
    profitable = oos[oos['total_return'] > 0]
    print(f'1. PROFITABILITY')
    print(f'   {len(profitable)}/{len(oos)} ({len(profitable)/len(oos)*100:.0f}%) of OOS configs are profitable')
    print(f'   Median OOS Sharpe: {oos["sharpe_ratio"].median():.2f}')
    print(f'   Best OOS Sharpe: {oos["sharpe_ratio"].max():.2f}')
    print()
    
    # 2. CEX-DEX vs CEX-CEX
    if 'pair_type' not in oos.columns:
        oos['pair_type'] = oos.apply(
            lambda r: 'CEX-CEX' if EXCHANGES.get(r.get('exchange_a'), {}).get('type') == 'CEX' 
                      and EXCHANGES.get(r.get('exchange_b'), {}).get('type') == 'CEX'
            else 'DEX-DEX' if EXCHANGES.get(r.get('exchange_a'), {}).get('type') == 'DEX'
                      and EXCHANGES.get(r.get('exchange_b'), {}).get('type') == 'DEX'
            else 'CEX-DEX', axis=1
        )
    
    print(f'2. PAIR TYPE COMPARISON')
    for pt in ['CEX-CEX', 'CEX-DEX', 'DEX-DEX']:
        pt_data = oos[oos['pair_type'] == pt]
        if not pt_data.empty:
            print(f'   {pt}: median Sharpe={pt_data["sharpe_ratio"].median():.2f}, '
                  f'median return={pt_data["total_return"].median()*100:.1f}%')
    print()
    
    # 3. Risk
    print(f'3. RISK')
    print(f'   Median max drawdown: {oos["max_drawdown"].median()*100:.2f}%')
    print(f'   Worst max drawdown: {oos["max_drawdown"].max()*100:.2f}%')
    print()
    
    # 4. Market neutrality
    print(f'4. MARKET NEUTRALITY')
    print(f'   Strategy is designed to be market-neutral (long + short perp)')
    print(f'   Correlation with BTC: see NB 06 for analysis')
    print()
    
    # 5. Limitations
    print(f'5. LIMITATIONS')
    print(f'   - Binance and Bybit geo-restricted (not in dataset)')
    print(f'   - Market impact not modeled (constant slippage assumption)')
    print(f'   - Fee schedules assumed constant over backtest period')
    print(f'   - Cross-exchange capital transfer delays not modeled')